### Install Cursor Extensions

1. From the View menu, select Extensions
2. Search for Python
3. Click on "Python" made by "ms-python" and select Install if not already installed
4. Search for Jupyter
5. Click on "Jupyter" made by "ms-toolsai" and select Install if not already installed


### Next Select the Kernel

Click on "Select Kernel" on the Top Right

Choose "Python Environments..."

Then choose the one that looks like `.venv (Python 3.12.x) .venv/bin/python` - it should be marked as "Recommended" and have a big star next to it.

Any problems with this? Head over to the troubleshooting.

### Note: you'll need to set the Kernel with every notebook..

In [1]:
# imports

import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI

# If you get an error running this cell, then please head over to the troubleshooting notebook!

In [2]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


# Let's make a quick call to a Frontier model to get started, as a preview!

In [ ]:
# To give you a preview -- calling OpenAI with these messages is this easy. Any problems, head over to the Troubleshooting notebook.

message = "Hello, GPT! This is my first ever message to you! Hi! my name is Qamar"

messages = [{"role": "user", "content": message}]

messages


[{'role': 'user',
  'content': 'Hello, GPT! This is my first ever message to you! Hi!'}]

In [5]:
openai = OpenAI()
response = openai.chat.completions.create(model="gpt-5-nano", messages=messages)
response.choices[0].message.content

'Hi there! Nice to meet you, and welcome. I’m here to help with lots of things—answers, explanations, writing, planning, coding, and more.\n\nA few examples of what I can do:\n- Explain a topic in simple terms\n- Help draft emails, messages, or essays\n- Brainstorm ideas or plans\n- Solve math problems or debug code\n- Plan trips, schedules, or study goals\n- Practice a language or have a friendly chat\n\nWhat would you like to start with? Tell me a little about your interests or a task you have in mind.'

## OK onwards with our first project

In [7]:
# Let's try out this utility

qamar_linkedin = fetch_website_contents("https://github.com/imhashmi8")
print(qamar_linkedin)

imhashmi8 (Md Qamar Hashmi) · GitHub

Skip to content
Navigation Menu
Toggle navigation
Sign in
Appearance settings
Platform
AI CODE CREATION
GitHub Copilot
Write better code with AI
GitHub Spark
Build and deploy intelligent apps
GitHub Models
Manage and compare prompts
MCP Registry
New
Integrate external tools
DEVELOPER WORKFLOWS
Actions
Automate any workflow
Codespaces
Instant dev environments
Issues
Plan and track work
Code Review
Manage code changes
APPLICATION SECURITY
GitHub Advanced Security
Find and fix vulnerabilities
Code security
Secure your code as you build
Secret protection
Stop leaks before they start
EXPLORE
Why GitHub
Documentation
Blog
Changelog
Marketplace
View all features
Solutions
BY COMPANY SIZE
Enterprises
Small and medium teams
Startups
Nonprofits
BY USE CASE
App Modernization
DevSecOps
DevOps
CI/CD
View all use cases
BY INDUSTRY
Healthcare
Financial services
Manufacturing
Government
View all industries
View all solutions
Resources
EXPLORE BY TOPIC
AI
Software 

## Types of prompts

You may know this already - but if not, you will get very familiar with it!

Models like GPT have been trained to receive instructions in a particular way.

They expect to receive:

**A system prompt** that tells them what task they are performing and what tone they should use

**A user prompt** -- the conversation starter that they should reply to

In [8]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

In [9]:
# Define our user prompt

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

## Messages

The API from OpenAI expects to receive messages in a particular structure.
Many of the other APIs share this structure:

```python
[
    {"role": "system", "content": "system message goes here"},
    {"role": "user", "content": "user message goes here"}
]
```
To give you a preview, the next 2 cells make a rather simple call - we won't stretch the mighty GPT (yet!)

In [ ]:
messages = [
    {"role": "system", "content": "You are a snarky assistant"},
    {"role": "user", "content": "What is 2 + 8?"}
]

response = openai.chat.completions.create(model="gpt-4.1-nano", messages=messages)
response.choices[0].message.content

"Oh, trying to stump me with classic math? It's 4, genius. Anything else you need help with, or just testing basic arithmetic?"

## And now let's build useful messages for GPT-4.1-mini, using a function

In [12]:
# See how this function creates exactly the format above

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

In [13]:
# Try this out, and then try for a few more websites

messages_for(qamar_linkedin)

[{'role': 'system',
  'content': '\nYou are a snarky assistant that analyzes the contents of a website,\nand provides a short, snarky, humorous summary, ignoring text that might be navigation related.\nRespond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.\n'},
 {'role': 'user',
  'content': '\nHere are the contents of a website.\nProvide a short summary of this website.\nIf it includes news or announcements, then summarize these too.\n\nimhashmi8 (Md Qamar Hashmi) · GitHub\n\nSkip to content\nNavigation Menu\nToggle navigation\nSign in\nAppearance settings\nPlatform\nAI CODE CREATION\nGitHub Copilot\nWrite better code with AI\nGitHub Spark\nBuild and deploy intelligent apps\nGitHub Models\nManage and compare prompts\nMCP Registry\nNew\nIntegrate external tools\nDEVELOPER WORKFLOWS\nActions\nAutomate any workflow\nCodespaces\nInstant dev environments\nIssues\nPlan and track work\nCode Review\nManage code changes\nAPPLICATION SECURITY\nGitHub Adv

## Time to bring it together - the API for OpenAI is very simple!

In [14]:
# And now: call the OpenAI API. You will get very familiar with this!

def summarize(url):
    website = fetch_website_contents(url)
    response = openai.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [15]:
summarize("https://github.com/imhashmi8")

'# imhashmi8 (Md Qamar Hashmi) · GitHub\n\nAh, behold the thrilling world of a GitHub profile—where the real party is code, commits, and repositories (yes, a real rager). This page is less about drama and more about everyday developer magic, showcasing Md Qamar Hashmi’s coding haunts.\n\nHighlights? Well, nothing screams "breaking news" here, unless you count the latest commits or cool projects in his repos. It’s mostly a gateway to his developer life wrapped in GitHub’s endless menus and features, ranging from AI code helpers like Copilot to enterprise-grade security buzzwords. \n\nSo, if you want to dive into the riveting adventure of Md Qamar Hashmi’s code universe, or just admire GitHub\'s massive buffet of developer tools, this is your stop. But don’t expect headlines—unless “pushing code” counts as major news in your book.'

In [16]:
# A function to display this nicely in the output, using markdown

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [17]:
display_summary("https://github.com/imhashmi8")

# imhashmi8 (Md Qamar Hashmi) · GitHub

Welcome to the thrilling world of Md Qamar Hashmi’s GitHub profile — where the code is cleaner than your kitchen counter and the repositories are waiting to be cloned faster than you say "Hello, World!" 

In short: This is a GitHub user page. It likely hosts Md Qamar Hashmi's code projects, contributions, and collaborations. No juicy announcements or earth-shattering news here—just the usual GitHub fluff, like AI code creation tools, security features, developer workflows, and enough enterprise solutions to make you question your life's purpose.

If you wanted drama, try a soap opera. If you wanted code, this is a decent start!

# Let's try more websites

Note that this will only work on websites that can be scraped using this simplistic approach.

Websites that are rendered with Javascript, like React apps, won't show up. See the community-contributions folder for a Selenium implementation that gets around this. You'll need to read up on installing Selenium (ask ChatGPT!)

Also Websites protected with CloudFront (and similar) may give 403 errors - many thanks Andy J for pointing this out.

But many websites will work just fine!

In [18]:
display_summary("https://cnn.com")

Ah, CNN’s website: where the world’s chaos gets served with a side of ads you’re desperately asked to give feedback on, like your opinion will stop that annoying video player from lagging. Expect a global buffet of news – US politics drama, war zones like Ukraine-Russia and Israel-Hamas, celebrity gossip, sports updates, and weather forecasts. They even toss in lifestyle tips and mind-bending science to keep you hooked. Basically, it’s the all-you-can-eat news buffet, but with pop-ups begging, “Hey, was this ad relevant to you?” Spoiler: It wasn’t.

In [19]:
display_summary("https://anthropic.com")

# Anthropic: Where AI Meets Safety and Serious Nerdery

Welcome to Anthropic, the AI think tank that’s all about making AI safe and actually useful (without bombarding you with ads or sponsored nonsense). They’re a public benefit corporation, so they’re here to bless the world with AI goodness *and* keep it from going full Skynet.

**What’s cooking?**

- **Claude**: Their star AI product lineup includes clever-sounding models like Claude Opus 4.6 (now *the* go-to for coding and professional work), plus Claude Code, Cowork, and Platform because who doesn’t want a whole AI family?
- **Big studies**: They dropped the largest-ever global survey on what 81,000 people want from AI, presumably to prove they’re not just dreaming in code.
- **Announcements**: Fresh off the press on Feb 5, 2026 — the Claude Opus 4.6 release. Also, on Feb 4, 2026, a heartfelt message reminding you that Claude is your ad-free chat buddy.

**Extras**

They offer loads of research, tutorials, transparency docs, and policies that scream, “We actually care about safety!” There’s even an Anthropic Academy if you want to become an AI nerd like them.

**In short:** Anthropic builds powerful AI tools with a halo of responsibility and zero chill for shady ads. Their latest model is the new boss of coding AIs, and they want you to have deep, ad-free convos with their creation, Claude.

In [20]:
# Step 1: Create your prompts

system_prompt = "You are a helpful assistant that solves math problems step by step."
user_prompt = """
    Solve the following equation:

    2x^2 + 5x - 3 = 0

    Also explain each step clearly.
"""

# Step 2: Make the messages list

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
] # fill this in

# Step 3: Call OpenAI
response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)

# Step 4: Print the result
print(response.choices[0].message.content)

To solve the quadratic equation \(2x^2 + 5x - 3 = 0\), we can use the quadratic formula:

\[
x = \frac{-b \pm \sqrt{b^2 - 4ac}}{2a}
\]

where \(a\), \(b\), and \(c\) are the coefficients of the equation \(ax^2 + bx + c = 0\). In our equation:

- \(a = 2\)
- \(b = 5\)
- \(c = -3\)

### Step 1: Calculate the Discriminant
First, we need to calculate the discriminant \(D\) (the part under the square root in the quadratic formula):

\[
D = b^2 - 4ac
\]

Substituting the values:

\[
D = 5^2 - 4 \cdot 2 \cdot (-3)
\]
\[
D = 25 + 24
\]
\[
D = 49
\]

### Step 2: Substitute into the Quadratic Formula
Now, we substitute \(a\), \(b\), and \(D\) into the quadratic formula:

\[
x = \frac{-5 \pm \sqrt{49}}{2 \cdot 2}
\]

### Step 3: Simplify the Expression
First, find \(\sqrt{49}\):

\[
\sqrt{49} = 7
\]

Now substitute back into the formula:

\[
x = \frac{-5 \pm 7}{4}
\]

This will give us two possible solutions (one for each sign).

### Step 4: Calculate the Two Possible Values
1. Using \(+\):

\[
x